# BERT Embedding & Visualization
**GNN-based Fraud Review Detection | SKK DScover**

---
## 목표
- `BAAI/bge-large-en-v1.5` 모델로 리뷰 텍스트 임베딩 (1024 dim)
- 임베딩 저장 → GNN 노드 피처로 활용
- t-SNE / UMAP 시각화로 사기/정상 리뷰 분포 확인

## 설치
```bash
pip install sentence-transformers umap-learn
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import torch
import os
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 8)
sns.set_theme(style='whitegrid')

DATA_PATH  = '../data/subgraph.csv'
EMBED_PATH = '../data/bert_embeddings.npy'
MODEL_NAME = 'BAAI/bge-large-en-v1.5'
BATCH_SIZE = 128
RANDOM_STATE = 42

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 1. 데이터 로드

In [ ]:
df = pd.read_csv(DATA_PATH, index_col=0, parse_dates=['date'])
df = df.reset_index(drop=True)

print(f'shape: {df.shape}')
print(f'label: {df["label"].value_counts().to_dict()}')

texts = df['text'].fillna('').tolist()
print(f'\n임베딩할 리뷰 수: {len(texts):,}')

## 2. BGE 임베딩

> **`BAAI/bge-large-en-v1.5`** — MTEB 영어 벤치마크 최상위권, 1024 dim
> BGE 모델은 `normalize_embeddings=True` 필수

In [ ]:
# 이미 임베딩 파일이 있으면 로드, 없으면 새로 생성
if os.path.exists(EMBED_PATH):
    print(f'기존 임베딩 로드: {EMBED_PATH}')
    embeddings = np.load(EMBED_PATH)
    print(f'shape: {embeddings.shape}')
else:
    from sentence_transformers import SentenceTransformer

    print(f'모델 로드 중: {MODEL_NAME}')
    model = SentenceTransformer(MODEL_NAME, device=device)

    print(f'임베딩 시작 (batch_size={BATCH_SIZE}, device={device})...')
    embeddings = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True
    )

    np.save(EMBED_PATH, embeddings)
    size_mb = os.path.getsize(EMBED_PATH) / (1024 ** 2)
    print(f'\n저장 완료: {EMBED_PATH} ({size_mb:.1f} MB)')
    print(f'임베딩 shape: {embeddings.shape}')

In [ ]:
print(f'임베딩 차원   : {embeddings.shape[1]}')
print(f'노드 수       : {embeddings.shape[0]:,}')
print(f'L2 norm 확인  : {np.linalg.norm(embeddings[0]):.4f}  (normalize=True 이면 ~1.0)')

## 3. t-SNE 시각화

전체 48k 노드를 t-SNE로 처리하면 매우 느리므로 **5,000개 샘플링** 후 시각화

In [ ]:
from sklearn.manifold import TSNE

N_SAMPLE = 5000

np.random.seed(RANDOM_STATE)
sample_idx = np.random.choice(len(df), N_SAMPLE, replace=False)

X_sample = embeddings[sample_idx]
y_sample = df['label'].values[sample_idx]

print(f't-SNE 실행 중 ({N_SAMPLE}개 샘플)...')
tsne = TSNE(n_components=2, random_state=RANDOM_STATE,
            perplexity=30, n_iter=1000, n_jobs=-1)
X_tsne = tsne.fit_transform(X_sample)
print('완료')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 9))

colors = {0: '#2196F3', 1: '#F44336'}
labels_name = {0: 'Normal', 1: 'Fraud'}
alphas = {0: 0.3, 1: 0.7}
sizes = {0: 8, 1: 15}

for label in [0, 1]:
    mask = y_sample == label
    ax.scatter(
        X_tsne[mask, 0], X_tsne[mask, 1],
        c=colors[label], label=labels_name[label],
        alpha=alphas[label], s=sizes[label], linewidths=0
    )

ax.set_title(f't-SNE Visualization of Review Embeddings\n'
             f'(BGE-large-en-v1.5, {N_SAMPLE} samples)', fontsize=14)
ax.set_xlabel('t-SNE dim 1')
ax.set_ylabel('t-SNE dim 2')
ax.legend(fontsize=12, markerscale=2)
plt.tight_layout()
plt.savefig('../data/tsne_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('저장: ../data/tsne_visualization.png')

## 4. UMAP 시각화

t-SNE보다 빠르고 전역 구조 보존이 우수

In [ ]:
try:
    import umap

    print('UMAP 실행 중...')
    reducer = umap.UMAP(n_components=2, random_state=RANDOM_STATE,
                        n_neighbors=15, min_dist=0.1)
    X_umap = reducer.fit_transform(X_sample)
    print('완료')

    fig, ax = plt.subplots(figsize=(12, 9))
    for label in [0, 1]:
        mask = y_sample == label
        ax.scatter(
            X_umap[mask, 0], X_umap[mask, 1],
            c=colors[label], label=labels_name[label],
            alpha=alphas[label], s=sizes[label], linewidths=0
        )
    ax.set_title(f'UMAP Visualization of Review Embeddings\n'
                 f'(BGE-large-en-v1.5, {N_SAMPLE} samples)', fontsize=14)
    ax.set_xlabel('UMAP dim 1')
    ax.set_ylabel('UMAP dim 2')
    ax.legend(fontsize=12, markerscale=2)
    plt.tight_layout()
    plt.savefig('../data/umap_visualization.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('저장: ../data/umap_visualization.png')

except ImportError:
    print('UMAP 미설치: pip install umap-learn')

## 5. 임베딩 통계 분석

In [ ]:
# 사기 vs 정상 임베딩 평균 코사인 유사도
fraud_idx = np.where(df['label'].values == 1)[0]
normal_idx = np.where(df['label'].values == 0)[0]

fraud_mean  = embeddings[fraud_idx].mean(axis=0)
normal_mean = embeddings[normal_idx].mean(axis=0)

cosine_sim = np.dot(fraud_mean, normal_mean)  # normalize=True 이므로 이미 unit vector

print('=== 임베딩 통계 ===')
print(f'사기 리뷰 평균 벡터 norm : {np.linalg.norm(fraud_mean):.4f}')
print(f'정상 리뷰 평균 벡터 norm : {np.linalg.norm(normal_mean):.4f}')
print(f'사기 vs 정상 평균 코사인 유사도: {cosine_sim:.4f}')
print(f'  → 1에 가까울수록 유사, 0에 가까울수록 구분 가능')

## 다음 단계
- `03_graph_construction.ipynb` — BERT 임베딩을 노드 피처로 사용하여 그래프 재구성
- `05_gnn_modeling.ipynb` — GCN / GAT / GraphSAGE 모델 학습